# Tugas 3 — Ekstraksi Dokumen Putusan PN Tarakan (2023–2025)
## Dataset Narkotika dan Psikotropika

Notebook ini disusun berdasarkan dua berkas yang disediakan:
- `Narkotika.zip` — berisi PDF putusan dalam folder tahun 2023, 2024, dan 2025.
- `Overview.xlsx` — daftar nomor putusan dan kolom metadata kelompok.

**Penting:** Akses otomatis ke Direktori Putusan MA dari Google Colab sebelumnya mendapat HTTP 403. Karena itu notebook ini mengolah PDF yang sudah diunduh. Notebook ini melakukan ekstraksi dokumen secara otomatis, bukan mengklaim berhasil melakukan crawling langsung ke situs MA. Jika dosen meminta proses crawling web, jelaskan kendala akses 403 dan dokumentasikan proses pengambilan PDF yang telah dilakukan.


## 1. Instalasi dan import library

In [1]:
!pip -q install pymupdf pandas openpyxl tqdm

import re
import zipfile
import shutil
from pathlib import Path

import pandas as pd
import fitz
from tqdm.auto import tqdm
from google.colab import files

print("Library berhasil disiapkan.")


Library berhasil disiapkan.


/usr/local/lib/python3.13/dist-packages/fitz/__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


## 2. Upload dua file dataset

In [2]:
uploaded = files.upload()

zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
xlsx_name = next((name for name in uploaded if name.lower().endswith(".xlsx")), None)

if zip_name is None:
    raise ValueError("File Narkotika.zip belum diunggah.")
if xlsx_name is None:
    raise ValueError("File Overview.xlsx belum diunggah.")

print("ZIP:", zip_name)
print("Excel:", xlsx_name)


Saving Narkotika.zip to Narkotika.zip
Saving Overview.xlsx to Overview.xlsx
ZIP: Narkotika.zip
Excel: Overview.xlsx


## 3. Ekstrak ZIP dan periksa jumlah PDF

In [3]:
ROOT = Path("/content/Tugas3_PN_Tarakan")
PDF_ROOT = ROOT / "PDF_Asli"
TXT_DIR = ROOT / "Dataset_TXT"
LOG_DIR = ROOT / "Log"

# Bersihkan hasil proses sebelumnya agar PDF tidak terhitung ganda
for folder in [PDF_ROOT, TXT_DIR, LOG_DIR]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

# Ekstrak file ZIP
with zipfile.ZipFile(zip_name, "r") as arsip:
    arsip.extractall(PDF_ROOT)

pdf_files = sorted(PDF_ROOT.rglob("*.pdf"))

print("Jumlah PDF ditemukan:", len(pdf_files))
print("Jumlah PDF per tahun:")

for tahun in ["2023", "2024", "2025"]:
    jumlah = sum(tahun in str(path) for path in pdf_files)
    print(tahun, ":", jumlah)

if len(pdf_files) == 0:
    raise FileNotFoundError("Tidak ditemukan PDF di dalam ZIP.")

if len(pdf_files) != 50:
    print("PERHATIAN: jumlah PDF tidak sama dengan target 50.")


Jumlah PDF ditemukan: 50
Jumlah PDF per tahun:
2023 : 15
2024 : 15
2025 : 20


## 4. Membaca Overview.xlsx

In [4]:
overview = pd.read_excel(xlsx_name)
overview.columns = [str(kolom).strip() for kolom in overview.columns]

if "No Putusan" not in overview.columns:
    raise ValueError("Kolom 'No Putusan' tidak ditemukan di Overview.xlsx.")

overview = overview.dropna(subset=["No Putusan"]).copy()
overview["No Putusan"] = overview["No Putusan"].astype(str).str.strip()
overview = overview[overview["No Putusan"] != ""]
overview = overview.drop_duplicates(subset=["No Putusan"]).reset_index(drop=True)

def normalisasi_nomor(teks):
    return re.sub(r"[^A-Z0-9]", "", str(teks).upper())

overview["Nomor_Normal"] = overview["No Putusan"].apply(normalisasi_nomor)

print("Jumlah nomor putusan unik di Overview:", len(overview))
display(overview[["No Putusan", "Lembaga Peradilan"]].head(10))


Jumlah nomor putusan unik di Overview: 50


,No Putusan,Lembaga Peradilan
0,163/Pid.Sus/2023/PN Tar,PN TARAKAN
1,247/Pid.Sus/2023/PN Tar,PN TARAKAN
2,273/Pid.Sus/2023/PN Tar,PN TARAKAN
3,274/Pid.Sus/2023/PN Tar,PN TARAKAN
4,257/Pid.Sus/2023/PN Tar,PN TARAKAN
5,254/Pid.Sus/2023/PN Tar,PN TARAKAN
6,264/Pid.Sus/2023/PN Tar,PN TARAKAN
7,291/Pid.Sus/2023/PN Tar,PN TARAKAN
8,252/Pid.Sus/2023/PN Tar,PN TARAKAN
9,299/Pid.Sus/2023/PN Tar,PN TARAKAN


## 5. Fungsi ekstraksi teks dan nomor putusan

In [5]:
def ekstrak_teks_pdf(path_pdf):
    """Mengambil teks dari semua halaman PDF."""
    bagian_teks = []

    with fitz.open(path_pdf) as dokumen:
        for nomor_halaman, halaman in enumerate(dokumen, start=1):
            isi = halaman.get_text("text")
            bagian_teks.append(
                f"\n===== HALAMAN {nomor_halaman} =====\n{isi}"
            )

    return "\n".join(bagian_teks).strip()


def rapikan_teks(teks):
    teks = teks.replace("\x00", " ")
    teks = re.sub(r"[ \t]+", " ", teks)
    teks = re.sub(r"\n{3,}", "\n\n", teks)
    return teks.strip()


POLA_NOMOR = re.compile(
    r"\b\d+\s*/\s*Pid\.?\s*Sus\.?\s*/\s*\d{4}\s*/\s*PN\s*Tar\b",
    flags=re.IGNORECASE
)


def ekstrak_nomor_putusan(teks, nama_file):
    # Nomor perkara dicari dari isi PDF
    cocok = POLA_NOMOR.search(teks)

    if cocok:
        return re.sub(r"\s+", "", cocok.group(0))

    # Jika tidak ditemukan, ambil pola dari nama file
    pola_nama = re.search(
        r"(\d+)\s*Pid\.?\s*Sus\.?\s*(\d{4})\s*PN\s*Tar",
        nama_file,
        flags=re.IGNORECASE
    )

    if pola_nama:
        return f"{pola_nama.group(1)}/Pid.Sus/{pola_nama.group(2)}/PN Tar"

    return "Nomor tidak ditemukan"


## 6. Fungsi mengambil amar putusan dan barang bukti

In [6]:
def ekstrak_amar_putusan(teks):
    """Ambil bagian amar mulai heading MENGADILI sampai sebelum penutup putusan."""
    pola_mengadili = re.compile(
        r"M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I\s*[:：]?",
        flags=re.IGNORECASE
    )

    semua_heading = list(pola_mengadili.finditer(teks))

    if not semua_heading:
        return ""

    # Heading terakhir adalah bagian amar putusan
    awal = semua_heading[-1].start()
    bagian = teks[awal:]

    penutup = re.search(
        r"\n\s*Demikian(?:lah)?\s+diputuskan",
        bagian,
        flags=re.IGNORECASE
    )

    if penutup:
        bagian = bagian[:penutup.start()]

    return bagian.strip()


def ekstrak_barang_bukti(amar):
    """Ambil daftar barang bukti dari bagian amar putusan."""
    pola_awal = re.search(
        r"(?:Menyatakan|Menetapkan)\s+barang\s+bukti\s+berupa\s*[:：]?",
        amar,
        flags=re.IGNORECASE
    )

    if not pola_awal:
        return ""

    sisa = amar[pola_awal.end():]

    # Berhenti pada butir amar utama berikutnya, biasanya biaya perkara.
    pola_akhir = re.search(
        r"\n\s*\d+\s*[.)]\s*"
        r"(?:Menetapkan|Membebankan|Menghukum|Memerintahkan|Menyatakan)\b",
        sisa,
        flags=re.IGNORECASE
    )

    if pola_akhir:
        sisa = sisa[:pola_akhir.start()]

    return sisa.strip()


## 7. Memproses seluruh PDF

In [7]:
hasil = []
log_proses = []

for path_pdf in tqdm(pdf_files, desc="Mengekstrak 50 PDF"):
    try:
        teks_asli = ekstrak_teks_pdf(path_pdf)
        teks = rapikan_teks(teks_asli)

        nomor = ekstrak_nomor_putusan(teks, path_pdf.stem)
        amar = ekstrak_amar_putusan(teks)
        barang_bukti = ekstrak_barang_bukti(amar)

        # Simpan hasil ekstraksi teks ke TXT
        path_txt = TXT_DIR / (path_pdf.stem + ".txt")
        path_txt.write_text(teks_asli, encoding="utf-8")

        # Tentukan tahun dari folder asal
        tahun = next(
            (th for th in ["2023", "2024", "2025"] if th in str(path_pdf)),
            ""
        )

        hasil.append({
            "No Putusan": nomor,
            "Lembaga Peradilan": "Pengadilan Negeri Tarakan",
            "Barang Bukti": barang_bukti if barang_bukti else "Tidak ditemukan",
            "Amar Putusan": amar if amar else "Tidak ditemukan",
            "Tahun": tahun,
            "Nama File PDF": path_pdf.name,
            "Nama File TXT": path_txt.name,
            "Jumlah Karakter": len(teks)
        })

        log_proses.append({
            "Nama File PDF": path_pdf.name,
            "Status": "Berhasil",
            "Jumlah Halaman": len(fitz.open(path_pdf)),
            "Jumlah Karakter": len(teks),
            "Nomor Ditemukan": nomor != "Nomor tidak ditemukan",
            "Amar Ditemukan": bool(amar),
            "Barang Bukti Ditemukan": bool(barang_bukti)
        })

    except Exception as error:
        log_proses.append({
            "Nama File PDF": path_pdf.name,
            "Status": f"Gagal: {error}",
            "Jumlah Halaman": 0,
            "Jumlah Karakter": 0,
            "Nomor Ditemukan": False,
            "Amar Ditemukan": False,
            "Barang Bukti Ditemukan": False
        })

hasil_df = pd.DataFrame(hasil)
log_df = pd.DataFrame(log_proses)

print("Jumlah PDF:", len(pdf_files))
print("Jumlah dokumen diproses:", len(hasil_df))
print("Jumlah TXT:", len(list(TXT_DIR.glob("*.txt"))))
display(hasil_df[[
    "No Putusan", "Tahun", "Barang Bukti", "Amar Putusan"
]].head(5))


Mengekstrak 50 PDF:   0%|          | 0/50 [00:00<?, ?it/s]

Jumlah PDF: 50
Jumlah dokumen diproses: 50
Jumlah TXT: 50


,No Putusan,Tahun,Barang Bukti,Amar Putusan
0,163/Pid.Sus/2023/PNTar,2023,- \n29 (dua puluh sembilan) Bungkus Plastik Be...,M E N G A D I L I : \n1. Menyatakan Terdakwa R...
1,169/Pid.Sus/2023/PNTar,2023,- 1 (satu) buah alat bong; \n- 1 (satu) buah k...,MENGADILI:\n1.\nMenyatakan Terdakwa Herman Bin...
2,174/Pid.Sus/2023/PNTar,2023,-\n2 (dua) bungkus plastik bening berisikan na...,M E N G A D I L I :\n1. Menyatakan Terdakwa JU...
3,247/Pid.Sus/2023/PNTar,2023,-\n1 (satu) Bungkus Plastik Klip Bening Berisi...,MENGADILI:\n1.\nMenyatakan Terdakwa Salang Bin...
4,252/Pid.Sus/2023/PNTar,2023,\n1 (satu) bungkus plastik bening berisi kris...,MENGADILI:\n1. Menyatakan Terdakwa ANTONI Alia...


## 8. Mencocokkan nomor putusan dengan Overview.xlsx

In [8]:
hasil_df["Nomor_Normal"] = hasil_df["No Putusan"].apply(normalisasi_nomor)
nomor_overview = set(overview["Nomor_Normal"])
hasil_df["Cocok dengan Overview"] = hasil_df["Nomor_Normal"].isin(nomor_overview)

nomor_hasil = set(hasil_df["Nomor_Normal"])
belum_ditemukan = overview[
    ~overview["Nomor_Normal"].isin(nomor_hasil)
][["No Putusan"]].copy()

print("Nomor di Overview:", len(overview))
print("Nomor hasil ekstraksi:", len(hasil_df))
print("Nomor cocok:", int(hasil_df["Cocok dengan Overview"].sum()))
print("Nomor Overview yang belum ditemukan:", len(belum_ditemukan))

display(hasil_df[[
    "No Putusan", "Cocok dengan Overview"
]].head(10))
display(belum_ditemukan)


Nomor di Overview: 50
Nomor hasil ekstraksi: 50
Nomor cocok: 50
Nomor Overview yang belum ditemukan: 1


,No Putusan,Cocok dengan Overview
0,163/Pid.Sus/2023/PNTar,True
1,169/Pid.Sus/2023/PNTar,True
2,174/Pid.Sus/2023/PNTar,True
3,247/Pid.Sus/2023/PNTar,True
4,252/Pid.Sus/2023/PNTar,True
5,254/Pid.Sus/2023/PNTar,True
6,257/Pid.Sus/2023/PNTar,True
7,264/Pid.Sus/2023/PNTar,True
8,273/Pid.Sus/2023/PNTar,True
9,274/Pid.Sus/2023/PNTar,True


,No Putusan
18,303/Pid.Sus/2024/PN Tar


## 9. Menyimpan CSV dan laporan pemeriksaan

In [9]:
# CSV utama sesuai kolom yang diminta dalam tugas
kolom_utama = [
    "No Putusan",
    "Lembaga Peradilan",
    "Barang Bukti",
    "Amar Putusan"
]

csv_path = ROOT / "Hasil_Ekstraksi.csv"
hasil_df[kolom_utama].to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig"
)

# Simpan file audit/validasi terpisah
log_df.to_csv(
    LOG_DIR / "Log_Pemrosesan.pdf.csv",
    index=False,
    encoding="utf-8-sig"
)

hasil_df.drop(columns=["Nomor_Normal"], errors="ignore").to_csv(
    LOG_DIR / "Detail_Hasil_Ekstraksi.csv",
    index=False,
    encoding="utf-8-sig"
)

belum_ditemukan.to_csv(
    LOG_DIR / "Nomor_Belum_Ditemukan.csv",
    index=False,
    encoding="utf-8-sig"
)

print("CSV utama tersimpan:", csv_path)
print("\nRingkasan pemeriksaan:")
print("Nomor ditemukan:", int(log_df["Nomor Ditemukan"].sum()))
print("Amar ditemukan:", int(log_df["Amar Ditemukan"].sum()))
print("Barang bukti ditemukan:", int(log_df["Barang Bukti Ditemukan"].sum()))

display(hasil_df[kolom_utama].head(10))


CSV utama tersimpan: /content/Tugas3_PN_Tarakan/Hasil_Ekstraksi.csv

Ringkasan pemeriksaan:
Nomor ditemukan: 50
Amar ditemukan: 50
Barang bukti ditemukan: 50


,No Putusan,Lembaga Peradilan,Barang Bukti,Amar Putusan
0,163/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,- \n29 (dua puluh sembilan) Bungkus Plastik Be...,M E N G A D I L I : \n1. Menyatakan Terdakwa R...
1,169/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,- 1 (satu) buah alat bong; \n- 1 (satu) buah k...,MENGADILI:\n1.\nMenyatakan Terdakwa Herman Bin...
2,174/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,-\n2 (dua) bungkus plastik bening berisikan na...,M E N G A D I L I :\n1. Menyatakan Terdakwa JU...
3,247/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,-\n1 (satu) Bungkus Plastik Klip Bening Berisi...,MENGADILI:\n1.\nMenyatakan Terdakwa Salang Bin...
4,252/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,\n1 (satu) bungkus plastik bening berisi kris...,MENGADILI:\n1. Menyatakan Terdakwa ANTONI Alia...
5,254/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,\n1 (satu) bungkus plastik bening berisi kris...,MENGADILI:\n1. Menyatakan Terdakwa JUNANTO Ali...
6,257/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,- 1 (satu) bungkus plastic bening berukuran ke...,MENGADILI:\n1.\nMenyatakan Terdakwa Roni Subek...
7,264/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,-\n2 (dua) bungkus plastik bening berisi serbu...,M E N G A D I L I :\n1. Menyatakan Terdakwa AR...
8,273/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,1 (satu) bungkus plastic klip bening berisika...,MENGADILI:\n1.\nMenyatakan Terdakwa Jumadi Bin...
9,274/Pid.Sus/2023/PNTar,Pengadilan Negeri Tarakan,1 (satu) bungkus plastic klip bening berisika...,MENGADILI:\n1.\nMenyatakan Terdakwa Eko Prayet...


## 10. Membuat ZIP hasil dan mengunduh

In [10]:
zip_hasil = shutil.make_archive(
    str(ROOT / "Output_Tugas3_PN_Tarakan"),
    "zip",
    root_dir=ROOT
)

print("CSV:", csv_path)
print("ZIP:", zip_hasil)

files.download(str(csv_path))
files.download(zip_hasil)


CSV: /content/Tugas3_PN_Tarakan/Hasil_Ekstraksi.csv
ZIP: /content/Tugas3_PN_Tarakan/Output_Tugas3_PN_Tarakan.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Catatan validasi
- Dataset ZIP yang disediakan berisi 50 PDF, dan Overview.xlsx berisi 50 nomor putusan.
- Cocokkan kolom nomor putusan dan periksa apakah seluruh dokumen berasal dari PN Tarakan tahun 2023–2025.
- Buka sampel PDF dan bandingkan barang bukti serta amar putusan dengan CSV. Ekstraksi teks otomatis dapat dipengaruhi format dokumen.
- Kolom yang bertuliskan `Tidak ditemukan` harus diperiksa manual.
- Program ini melakukan ekstraksi dari PDF lokal. Proses crawling langsung ke website MA tidak dijalankan dalam notebook ini karena akses Colab sebelumnya memperoleh HTTP 403.
